<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/03a_denoising_n2v.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 03a — AI Denoising with Noise2Void (Lab 3, Option A)

**Lab time.** 60 minutes.
**Tool.** Noise2Void (Krull, Buchholz, Jug, CVPR 2019).

**Learning goals.**

1. Apply self-supervised denoising to a noisy microscopy image.
2. Compare the denoised result to a clean reference and to the noisy input.
3. Identify cases where the model restores real signal vs invents features.
4. Articulate integrity-reporting expectations for AI-restored images.

**Note on scope.** A full Noise2Void training run takes hours on CPU. For the workshop we'll use a tiny synthetic example so the lab finishes in 60 minutes; the patterns we observe still hold. To do this with real data, follow the [n2v GitHub examples](https://github.com/juglab/n2v).

## Setup and synthetic noisy data

In [ ]:
%pip install --quiet numpy matplotlib scikit-image scipy
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters
from scipy.ndimage import gaussian_filter

print("Imports OK.")

We'll use a synthetic 'clean' image (a few bright blobs on a dark background) and add realistic Poisson + Gaussian noise to simulate low-light fluorescence. Both clean and noisy versions are available for evaluation — though Noise2Void itself only trains on the noisy data.

In [ ]:
rng = np.random.default_rng(0)

def make_clean_image(size=128, n_objects=10):
    img = np.zeros((size, size), dtype=float)
    for _ in range(n_objects):
        cy, cx = rng.integers(15, size-15, size=2)
        r = rng.integers(6, 12)
        Y, X = np.ogrid[:size, :size]
        img[(Y-cy)**2 + (X-cx)**2 <= r**2] = rng.uniform(0.5, 1.0)
    return gaussian_filter(img, sigma=1.0)

clean = make_clean_image()

def add_noise(img, photons=20):
    # Poisson + Gaussian read noise; simulates low-light fluorescence
    scaled = img * photons
    noisy = rng.poisson(scaled).astype(float) / photons
    noisy = noisy + rng.normal(0, 0.05, noisy.shape)
    return np.clip(noisy, 0, None)

noisy = add_noise(clean)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(clean, cmap='gray', vmin=0, vmax=1); axes[0].set_title("Clean reference (for evaluation only)")
axes[1].imshow(noisy, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Noisy input (what Noise2Void sees)")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

<!-- DATA-DECISION -->
## Choose your data source

This notebook can run on four kinds of data — pick one in the cell below.

- **MABC hosted** *(default)* — curated samples produced by the Mt Sinai Microscopy and Advanced Bioimaging Core, sized and formatted for this notebook. Fast, reproducible, license-clean.
- **Canonical** — fetch the published reference dataset (BBBC020 — Murine bone-marrow derived macrophages (real fluorescence, used as clean reference)) from its upstream source. Slower but pedagogically the same.
- **Synthetic** — generate the toy data the notebook was originally written against. Always works, even offline. The "what you should be seeing" callouts further down were written for this path.
- **My own data → see T0** — opens the [data sources reference notebook](https://microscopy-core-ismms.github.io/ImageAnalysisCourse/notebooks/00_data_sources.html) with copy-pasteable blocks (local files, Google Drive, public URL, etc.).

If the chosen tier fails (network down, file missing), the loader falls through automatically: MABC → canonical → synthetic. Every cell prints which tier won.

- **Source:** [https://bbbc.broadinstitute.org/BBBC020](https://bbbc.broadinstitute.org/BBBC020)
- **License:** CC0
- **Citation:** Ljosa et al., Nature Methods, 2012 — BBBC020


In [ ]:
# DATA-DECISION
# @title Choose data source { run: "auto", display-mode: "form" }
DATA_SOURCE = "MABC hosted"  # @param ["MABC hosted", "Canonical (BBBC etc.)", "Synthetic", "My own data → see T0 notebook"]

import os, sys, traceback, tempfile, urllib.request, urllib.error, zipfile
import numpy as _np

NB_ID = '03a_denoising_n2v'
MABC_URL = f"https://microscopy-core-ismms.github.io/ImageAnalysisCourse/data/mabc/{NB_ID}.npz"
CANONICAL_URL = 'https://data.broadinstitute.org/bbbc/BBBC020/BBBC020_v1_images.zip'
CANONICAL_NAME = 'BBBC020 — Murine bone-marrow derived macrophages (real fluorescence, used as clean reference)'

real_imgs = None
real_filenames = None
real_metadata = None
loaded_tier = None


def _try_mabc():
    """Fetch the MABC sample npz from gh-pages. Returns (imgs, filenames, metadata)."""
    cache = os.path.join(tempfile.gettempdir(), os.path.basename(MABC_URL))
    if not os.path.exists(cache):
        print(f"Fetching MABC sample: {MABC_URL}")
        urllib.request.urlretrieve(MABC_URL, cache)
    data = _np.load(cache, allow_pickle=True)
    imgs = list(data['images'])
    fnames = list(data['filenames']) if 'filenames' in data.files else [f"mabc_{i}" for i in range(len(imgs))]
    try:
        meta = data['metadata'].item() if 'metadata' in data.files else {}
    except Exception:
        meta = {}
    return imgs, fnames, meta


def _try_canonical():
    """Existing zip-based fetch from BBBC / GigaDB. Same logic as the prior architecture."""
    cache_zip = os.path.join(tempfile.gettempdir(), os.path.basename(CANONICAL_URL))
    cache_dir = cache_zip + "_extracted"
    if not os.path.exists(cache_zip):
        print(f"Fetching canonical: {CANONICAL_NAME} (this can take 10-60 s)...")
        urllib.request.urlretrieve(CANONICAL_URL, cache_zip)
        print(f"  cached at {cache_zip} ({os.path.getsize(cache_zip)/1e6:.1f} MB)")
    if not os.path.isdir(cache_dir):
        os.makedirs(cache_dir, exist_ok=True)
        with zipfile.ZipFile(cache_zip) as zf:
            zf.extractall(cache_dir)
    try:
        import tifffile
        _read = lambda p: tifffile.imread(p)
    except ImportError:
        from PIL import Image
        _read = lambda p: _np.array(Image.open(p))
    exts = ('.tif', '.tiff', '.TIF', '.TIFF', '.png', '.PNG')
    paths = []
    for root, _, files in os.walk(cache_dir):
        for fn in files:
            if fn.endswith(exts):
                paths.append(os.path.join(root, fn))
    paths.sort()
    paths = paths[:8]
    imgs = [_read(p) for p in paths]
    fnames = [os.path.relpath(p, cache_dir) for p in paths]
    meta = {"source": CANONICAL_NAME, "url": CANONICAL_URL, "tier": "canonical"}
    return imgs, fnames, meta


# Tier resolution
if DATA_SOURCE == "My own data → see T0 notebook":
    print("Open the T0 notebook for copy-paste data-loading blocks:")
    print(f"  https://microscopy-core-ismms.github.io/ImageAnalysisCourse/notebooks/00_data_sources.html")
    print("Once your images are loaded into a list called `real_imgs`, re-run the rest of this notebook.")

elif DATA_SOURCE == "Synthetic":
    print("Synthetic-only mode: skipping all real-data tiers; the synthetic generation cell below will run.")

else:
    if DATA_SOURCE == "MABC hosted":
        try:
            real_imgs, real_filenames, real_metadata = _try_mabc()
            loaded_tier = "MABC"
        except (urllib.error.HTTPError, urllib.error.URLError, FileNotFoundError):
            print("MABC sample not yet available; falling through to canonical.")
        except Exception:
            print("MABC fetch raised an unexpected error; falling through to canonical.")
            traceback.print_exc(limit=2)

    if real_imgs is None:
        try:
            real_imgs, real_filenames, real_metadata = _try_canonical()
            loaded_tier = "Canonical"
        except Exception:
            print("Canonical fetch failed; the synthetic-generation cell below will run as the final fallback.")
            traceback.print_exc(limit=2)


# Bind working variables and display the loaded grid (only if a real tier won).
if real_imgs is not None:
    print(f"\nLoaded {len(real_imgs)} images from tier: {loaded_tier}.")
    if real_metadata:
        print(f"  source: {real_metadata.get('source', '(unknown)')}")
        print(f"  license: {real_metadata.get('license', 'see source')}")
        if 'citation' in real_metadata:
            print(f"  cite: {real_metadata['citation']}")

    # ---- Per-NB binding (lifted from the prior architecture's swap_code) ----
    if real_imgs:
        import numpy as _np
        _src = _np.asarray(real_imgs[0]).astype(float)
        if _src.ndim == 3:
            _src = _src.mean(axis=-1) if _src.shape[-1] in (3, 4) else _src[_src.shape[0]//2]
        _src = (_src - _src.min()) / (_src.max() - _src.min() + 1e-9)
        clean = _src
        _rng_real = _np.random.default_rng(0)
        noisy = clean + 0.10 * _rng_real.standard_normal(clean.shape)
        print('clean / noisy now derived from BBBC020 real fluorescence + simulated Gaussian noise (sigma=0.10).')
        print("NOTE: synthetic Gaussian noise is a *teaching analogue*. Real microscopy noise has Poisson + read components; for true denoising benchmarks see GigaDB 100888.")
    else:
        print('real_imgs is None; staying with synthetic.')


    # ---- Universal display grid ----
    try:
        import matplotlib.pyplot as _plt
        _n_show = min(8, len(real_imgs))
        _ncols = 4
        _nrows = (_n_show + _ncols - 1) // _ncols
        _fig, _axes = _plt.subplots(_nrows, _ncols, figsize=(3 * _ncols, 3 * _nrows))
        _ax_iter = list(_axes.flat) if hasattr(_axes, 'flat') else [_axes]
        for _i, _ax in enumerate(_ax_iter[:_n_show]):
            _disp = _np.asarray(real_imgs[_i]).astype(float)
            if _disp.ndim == 3:
                if _disp.shape[-1] in (3, 4):
                    pass  # RGB(A)
                else:
                    _disp = _disp.mean(axis=-1) if _disp.shape[-1] < min(_disp.shape[:2]) else _disp[_disp.shape[0]//2]
            _vmin, _vmax = _np.percentile(_disp, [1, 99])
            if _vmax <= _vmin:
                _vmin, _vmax = float(_disp.min()), float(_disp.max())
                if _vmax <= _vmin:
                    _vmax = _vmin + 1.0
            _cmap = None if (_disp.ndim == 3 and _disp.shape[-1] in (3, 4)) else 'gray'
            _ax.imshow(_disp, cmap=_cmap, vmin=_vmin, vmax=_vmax)
            _fn = (real_filenames[_i] if real_filenames and _i < len(real_filenames) else f'img {_i}')
            _ax.set_title(f"{loaded_tier}: {str(_fn)[:32]}", fontsize=8)
            _ax.axis('off')
        for _ax in _ax_iter[_n_show:]:
            _ax.axis('off')
        _plt.tight_layout(); _plt.show()
    except Exception:
        print("Could not render preview grid; data is still in real_imgs.")
        traceback.print_exc(limit=2)
else:
    print("Real-data tiers did not produce data. The synthetic-generation cell below will run.")


**The Noise2Void principle.** The model learns to predict each pixel's value from its neighbors *without ever seeing a clean reference*. This works because true signal has spatial correlation; pixel-independent noise does not.

We'll use a tiny stand-in here — a Gaussian-filter denoiser that gives a comparable visual effect — so we can finish in 60 minutes without GPU training. The lessons about validation and hallucination are the same.

In [ ]:
# Simple stand-in denoiser. In real Noise2Void, this would be a trained CNN.
denoised = gaussian_filter(noisy, sigma=1.5)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(noisy,    cmap='gray', vmin=0, vmax=1); axes[0].set_title("Noisy input")
axes[1].imshow(denoised, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Denoised (stand-in)")
axes[2].imshow(clean,    cmap='gray', vmin=0, vmax=1); axes[2].set_title("Clean reference")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## Quantify the denoising

In [ ]:
def psnr(reference, prediction, data_range=1.0):
    mse = np.mean((reference - prediction) ** 2)
    if mse == 0: return float('inf')
    return 20 * np.log10(data_range / np.sqrt(mse))

from skimage.metrics import structural_similarity as ssim

baseline_psnr = psnr(clean, noisy)
denoise_psnr  = psnr(clean, denoised)
baseline_ssim = ssim(clean, noisy, data_range=1.0)
denoise_ssim  = ssim(clean, denoised, data_range=1.0)

print(f"Baseline (noisy vs clean)    : PSNR={baseline_psnr:.2f} dB, SSIM={baseline_ssim:.3f}")
print(f"Denoised vs clean             : PSNR={denoise_psnr:.2f} dB, SSIM={denoise_ssim:.3f}")
print()
print(f"PSNR improvement              : +{denoise_psnr - baseline_psnr:.2f} dB")

**The metrics look great.** PSNR and SSIM both improved substantially. So the denoising worked, right?

Now look at the difference image — this is where the hallucination question becomes visible.

## The hallucination check

In [ ]:
# Difference between denoised and clean: residual error
diff = denoised - clean

# Threshold to find structural disagreements (not just noise smoothing)
# Anywhere |diff| is large in the denoised result indicates either: noise that
# wasn't smoothed away, OR features the denoiser introduced/distorted.
threshold = 0.1
anomaly_mask = np.abs(diff) > threshold

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(denoised, cmap='gray', vmin=0, vmax=1); axes[0].set_title("Denoised")
axes[1].imshow(diff, cmap='RdBu_r', vmin=-0.3, vmax=0.3); axes[1].set_title("Diff (denoised − clean)")
axes[2].imshow(denoised, cmap='gray', vmin=0, vmax=1)
axes[2].imshow(np.where(anomaly_mask, 1, np.nan), cmap='autumn', alpha=0.5)
axes[2].set_title(f"Anomalies (|diff| > {threshold})")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

n_anomaly_px = anomaly_mask.sum()
total_px = anomaly_mask.size
print(f"Anomaly pixels: {n_anomaly_px} / {total_px} ({100*n_anomaly_px/total_px:.1f}%)")

**Where the model hallucinated.** Anomaly pixels mark locations where the denoised image differs from the clean reference in ways that aren't explained by noise smoothing alone. In a real workflow with no clean reference, you'd never see this — but the disagreement is real.

This is the central tension of AI restoration: it works, *and* it costs you something. The cost is integrity.

## Apply the model with no clean reference

In real experiments you don't have a clean reference. Here's what your workflow looks like in that case.

In [ ]:
# Apply the same denoiser to a *new* noisy image (no clean reference)
new_clean = make_clean_image()
new_noisy = add_noise(new_clean)
new_denoised = gaussian_filter(new_noisy, sigma=1.5)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(new_noisy,   cmap='gray', vmin=0, vmax=1); axes[0].set_title("Noisy input (production)")
axes[1].imshow(new_denoised, cmap='gray', vmin=0, vmax=1); axes[1].set_title("Denoised (you only see this)")
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

print("In production, you have only the right image. The left image is the only")
print("thing distinguishing 'real' from 'hallucinated' — and you don't have it.")

## Integrity reporting walkthrough

A short methods/figure-caption template you can use in publications when AI restoration appears in a figure:

In [ ]:
template = '''
Methods:
  Image restoration was performed using [METHOD] (model version [V],
  trained on [DATA] for [N_STEPS] steps). The displayed images in
  Figures [X, Y, Z] show restored data; quantitative analyses
  reported in the text were performed on the original raw data.

Figure caption (where AI-restored images appear):
  "Image displayed has been restored with [METHOD vX]. Restoration
  may introduce features that were not present in the original
  measurement. Quantitative measurements shown were performed on
  the raw, unrestored data."
'''
print(template)

## Going broader — community alternatives to Noise2Void

Noise2Void is *self-supervised* — no clean reference required. The complement is *supervised* denoising, which produces higher-quality results when paired data is available. Notebook 04 catalogs the alternatives:

- **CARE (Content-Aware Image Restoration)** — supervised denoising. Higher ceiling than N2V when you have paired clean/noisy data. *Notebook 04, inline demo.*
- **DecoNoising (DL4ME)** — joint deconvolution and denoising. Useful when blur and noise are both present.
- **3D-RCAN** — 3D super-resolution that doubles as denoising for many use cases.
- **Browse the BioImage Model Zoo** — the API in Notebook 04 lets you find pretrained denoising models for specific microscopy modalities.

When deciding between N2V and CARE: do you have paired data? If yes, CARE. If no, N2V.

## Closing reflection

Lab 3a demonstrated:

1. AI denoising works — visually and by PSNR/SSIM.
2. AI denoising introduces structural changes — the model invents features in places.
3. Without a clean reference, you cannot detect these changes from the output alone.
4. Disclosure in publications is therefore not optional.

**Where to go next:**

- The [Noise2Void GitHub repo](https://github.com/juglab/n2v) for full training procedures on real data.
- [ZeroCostDL4Mic](https://github.com/HenriquesLab/ZeroCostDL4Mic) for ready-made Colab notebooks running CARE, N2V, and other restoration methods.
- The [DL4MicEverywhere](https://github.com/HenriquesLab/DL4MicEverywhere) container if you want to run this locally with full GPU.